# Chapter 17: unit roots, cointegration and forecasting

Fixed-seed mechanisms distinguish correlated drifting levels from a known stationary spread. Compare ordinary ADF with Engle–Granger calibration, then fit a bivariate VAR to the first 100 of 120 constructed months. The French cumulative-index screen is disabled. Historical index proxies are neither security prices nor PIT trading records.

[Entry modes and preparation](../NOTEBOOK_GUIDE.md). See this guide for run steps, data requirements, and expected outputs.

In [ ]:
from pathlib import Path
import sys, os, uuid
ROOT = next((p for p in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (p / "pyproject.toml").is_file() and (p / "src/quant_course").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Start inside the reader package tree")
sys.path.insert(0, str(ROOT / "src"))
sys.path.insert(0, str(ROOT / "code"))
os.environ["MPLBACKEND"] = "Agg"
RUN_DIR = ROOT / "runs" / ("ch17_time_series_ii_" + uuid.uuid4().hex)
RUN_DIR.mkdir(parents=True, exist_ok=False)
os.chdir(ROOT)
# French branches remain disabled until data rights and execution authorization are resolved.
RUN_FRENCH = False
print("CONSTRUCTED teaching route; French market branches SKIPPED; outputs:", RUN_DIR)

from pathlib import Path
import sys
import hashlib

import matplotlib
matplotlib.use("Agg")
import numpy as np
import pandas as pd
import statsmodels
from IPython.display import Image, display


from quant_course.cointegration import (
    adf_test,
    engle_granger_test,
    fit_var_forecast,
    screen_cointegration_pairs,
    simulate_cointegrated_pair,
    simulate_noncointegrated_common_trend,
    total_return_index_proxy,
    validate_monthly_frame,
)

print(f"Python={sys.version.split()[0]}; pandas={pd.__version__}; statsmodels={statsmodels.__version__}")

In [ ]:
# 固定种子构造：水平高相关但差仍是 I(1)，对照真实的已知平稳价差。
noncointegrated = simulate_noncointegrated_common_trend(n=900, seed=20261017)
cointegrated = simulate_cointegrated_pair(n=900, seed=20261018)
noncointegrated_eg = engle_granger_test(noncointegrated["x"], noncointegrated["y"])
cointegrated_eg = engle_granger_test(cointegrated["y"], cointegrated["x"])

print(f"共同趋势：corr(levels)={noncointegrated.x.corr(noncointegrated.y):.4f}; EG p={noncointegrated_eg.pvalue:.6f}")
print(f"协整构造：corr(levels)={cointegrated.x.corr(cointegrated.y):.4f}; EG p={cointegrated_eg.pvalue:.3e}")
print(f"协整 5% 专用临界值={cointegrated_eg.critical_values['5%']:.4f}")
assert noncointegrated.x.corr(noncointegrated.y) > 0.9
assert noncointegrated_eg.pvalue > 0.05
assert cointegrated_eg.pvalue < 0.01

## From a hand regression to statistical calibration

For $y_t=0.7y_{t-1}+\varepsilon_t$, subtraction gives $\Delta y_t=-0.3y_{t-1}+\varepsilon_t$. Thus ADF's simple AR(1) coefficient is $\gamma=\phi-1=-0.3$. ADF on the known spread and Engle–Granger on an estimated relation use different null calibration; their p-values are not interchangeable.

In [ ]:
gamma = 0.7 - 1.0
spread_adf = adf_test(cointegrated["spread"], regression="c")
print(f"手算 gamma={gamma:.1f}")
print(f"已知生成的平稳 spread：ADF stat={spread_adf.statistic:.4f}, p={spread_adf.pvalue:.3e}, lag={spread_adf.used_lag}")
print(f"估计关系的残差检验：EG stat={cointegrated_eg.statistic:.4f}, p={cointegrated_eg.pvalue:.3e}")

## Bivariate VAR: training and forecast months

Generate a stationary VAR(1), estimate its coefficients on 100 months and predict the next two recursively. Each coefficient-matrix row maps lagged variables to that row's target. Forecast dates start after the training-end month.

In [ ]:
rng = np.random.default_rng(1704)
values = np.zeros((120, 2), dtype=float)
A = np.array([[0.35, 0.12], [0.05, 0.30]])
c = np.array([0.25, -0.10])
for t in range(1, len(values)):
    values[t] = c + A @ values[t - 1] + rng.normal(0.0, 0.10, size=2)
index = pd.period_range("1990-01", periods=len(values), freq="M", name="month")
var_data = pd.DataFrame(values, index=index, columns=["industry_A", "industry_B"])
train = validate_monthly_frame(var_data.iloc[:100])
var_prediction = fit_var_forecast(train, lags=1, steps=2)
print("training end:", train.index[-1])
print(var_prediction)
assert var_prediction.index.tolist() == [train.index[-1] + 1, train.index[-1] + 2]
assert var_prediction.columns.tolist() == ["industry_A", "industry_B"]

## Disabled historical index screen

The optional loader would check a fixed processed SHA-256 and compound already decimal returns into index proxies. The 45-pair screen uses observations through December 2000 only. Nominal p-values and BH q-values have different meanings; neither selects a tradable winner here.

In [ ]:
if RUN_FRENCH:
    french_path = ROOT / "data/processed/french_10_industries_monthly.csv"
    expected_sha256 = "3b097d810c295d64eedfa1ece7067ff2a0707f5ca90e33935df9949251f70950"
    actual_sha256 = hashlib.sha256(french_path.read_bytes()).hexdigest()
    assert actual_sha256 == expected_sha256
    raw = pd.read_csv(french_path, dtype={"month": "string"})
    periods = pd.PeriodIndex(raw.pop("month"), freq="M", name="month")
    returns = validate_monthly_frame(pd.DataFrame(raw.to_numpy(dtype=float), index=periods, columns=raw.columns), min_observations=24)
    tr_index = total_return_index_proxy(returns)
    cutoff = pd.Period("2000-12", freq="M")
    pair_tests = screen_cointegration_pairs(tr_index, training_end=cutoff, trend="c")
    
    print(f"snapshot SHA-256={actual_sha256}")
    print(f"sample={returns.index[0]}–{returns.index[-1]}; n={len(returns)} months; pairs={len(pair_tests)}")
    print(f"training end={cutoff}; min p={pair_tests.pvalue.min():.6g}; median p={pair_tests.pvalue.median():.6g}")
    print(f"nominal p < .05={(pair_tests.pvalue < .05).sum()}; BH q < .05={(pair_tests.qvalue_bh < .05).sum()}")
    print("Exploratory current-vintage descriptive screen only; no selected pair is claimed tradable or profitable.")
    assert len(pair_tests) == 45
    assert pair_tests["training_end"].eq(str(cutoff)).all()
else:
    print("SKIPPED French branch: permissions unknown; no bundled data; historical empirical results not reproduced")


## Frozen constructed figure

Display the existing simulation PNG from fixtures/figures/a4. The top panels show levels, and lower panels reveal whether differences still drift. The French image is explicitly skipped; no new PNG is saved.

In [ ]:
display(Image(filename=str(ROOT / "fixtures/figures/a4/ch17_cointegration_simulation.png")))
print("SKIPPED French historical figure")


There is no strategy P&L, issuer-vintage panel, tick price, borrow record or fill. A month-end signal must await availability and trade later; both legs need the same post-fill return interval. The French cache cannot establish those facts.